# Module 02 — Lazy Execution

Transformations build a DAG; actions execute it.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import get_spark, ui_links
from shared import config

spark = get_spark('module-02-lazy-execution')
ui_links()

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())

## 1. Lazy Transformations

Build a chain — no jobs until an action.

In [ ]:
from pyspark.sql.functions import col

df = (trips
    .filter('total_amount > 0')
    .select('VendorID', 'PULocationID', 'total_amount', 'trip_distance', 'tpep_pickup_datetime')
    .filter('PULocationID IS NOT NULL')
    .withColumn('fare_per_mile', col('total_amount') / col('trip_distance'))
)
print('No jobs yet — check UI Jobs tab (should be empty or unchanged)')

In [ ]:
count = df.count()  # ACTION — triggers entire DAG
print(f'Row count: {count:,}')

## 2. Branching DAG — Duplicated Work

Reusing `df_filtered` in two actions scans the source twice.

In [ ]:
df_filtered = trips.filter('total_amount > 20')

c1 = df_filtered.count()   # Action 1 — full scan
c2 = df_filtered.filter('trip_distance > 5').count()  # Action 2 — scans df_filtered again
print(f'High fare trips: {c1:,}, long high-fare: {c2:,}')
# UI: two jobs, duplicated scan stages

## 3. Wasteful Patterns

`count()` + `filter().count()` on same base = repeated work.

In [ ]:
base = trips.filter('passenger_count > 0')
n1 = base.count()
n2 = base.filter('total_amount > 50').count()
print(f'Passengers>0: {n1:,}, Fare>50: {n2:,}')
# Both actions re-read and re-filter from source

## 4. Mitigation: Cache

In [ ]:
from pyspark import StorageLevel

base_cached = trips.filter('passenger_count > 0').persist(StorageLevel.MEMORY_AND_DISK)
base_cached.count()  # materializes cache

n1 = base_cached.count()  # fast — from cache
n2 = base_cached.filter('total_amount > 50').count()  # reads from cache
print(f'Cached: {n1:,}, Filtered: {n2:,}')
base_cached.unpersist()

## 5. Mitigation: Checkpoint

Breaks lineage by writing to reliable storage.

In [ ]:
spark.sparkContext.setCheckpointDir(str(config.CHECKPOINT_DIR))

cp = trips.filter('total_amount > 0').checkpoint()
cp.count()  # writes checkpoint, then counts
# Subsequent actions read from checkpoint, not full lineage

## 6. Spark Observation — Metrics Without Full Actions

In [ ]:
from pyspark.sql import Observation

obs = Observation('metrics')
observed = trips.filter('total_amount > 0').observe(obs, count('*'))
observed.count()  # single action collects observed metrics
print(obs.get)

## Key Takeaways

1. Only **actions** trigger execution.
2. Shared ancestors in branched DAGs cause **duplicate work** unless cached/checkpointed.
3. Use `.cache()`, `.checkpoint()`, and `Observation` strategically.